# M3 · Keypoints de llantas (YOLO11s-pose) y conteo de ejes
**IAM-811 · Laboratorio Final — Sistema Inteligente de Control de Peaje**
Basado en `keypoints_llantas_yolo_detectron2_v3.ipynb` (Ing. Alvaro Pinto Mamani).

No existe un modelo de keypoints de llantas listo, así que se sigue el flujo del docente:

| Etapa | Método |
|---|---|
| 1. Auto-etiquetado asistido | YOLOE-seg (prompts `wheel`, `tire`) + 5 keypoints geométricos del contorno |
| 2. Corrección humana | Roboflow, proyecto *Keypoint Detection* (esqueleto de 5 puntos) |
| 3. Entrenamiento | `yolo11s-pose.pt` con `kpt_shape: [5, 3]` |
| 4. Conteo de ejes | lado cercano + separación entre ejes + mínimo 2 ejes |

Keypoints por llanta: **0 centro · 1 contacto · 2 superior · 3 izquierda · 4 derecha** (`flip_idx = [0,1,2,4,3]`).
**Métricas:** mAP50 de pose (OKS) y **MAE del conteo de ejes**. **Salida:** `m3_llantas_pose.pt`.

## 1. Instalación

In [ ]:
!nvidia-smi -L || echo "ACTIVE LA GPU T4"
!pip -q install ultralytics==8.4.218 roboflow
!pip -q install 'git+https://github.com/ultralytics/CLIP.git'
import os, glob, random, shutil, json, yaml, cv2, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from ultralytics import YOLO, YOLOE
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'; print(DEVICE)

## 2. Utilidades del docente (zero-shot, limpieza y conteo de ejes)

In [ ]:
KPT_NOMBRES = ['centro', 'contacto', 'superior', 'izquierda', 'derecha']
FLIP_IDX = [0, 1, 2, 4, 3]
PROMPTS_LLANTA = ['wheel', 'tire', 'car wheel']
UMBRAL_CONTENCION, UMBRAL_CERCANA, SEPARACION_EJE = 0.7, 0.65, 0.6

def iou(a, b):
    x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    i = max(0, x2 - x1) * max(0, y2 - y1); u = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - i
    return i / u if u > 0 else 0

def contenida(a, b):
    x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    return max(0, x2 - x1) * max(0, y2 - y1) / max(1e-6, (a[2]-a[0]) * (a[3]-a[1]))

def nms_simple(items, u=0.5):
    res = []
    for d in sorted(items, key=lambda d: -d['conf']):
        if all(iou(d['box'], r['box']) < u for r in res): res.append(d)
    return res

_yoloe = None
def yoloe():
    global _yoloe
    if _yoloe is None:
        _yoloe = YOLOE('yoloe-11l-seg.pt'); _yoloe.set_classes(PROMPTS_LLANTA, _yoloe.get_text_pe(PROMPTS_LLANTA))
    return _yoloe

def kps_de_mascara(mask):
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts: return None
    c = max(cnts, key=cv2.contourArea)
    if cv2.contourArea(c) < 8: return None
    pts = c.reshape(-1, 2).astype(float)
    (cx, cy), _, _ = cv2.fitEllipse(c) if len(pts) >= 5 else (pts.mean(0), 0, 0)
    k = np.array([[cx, cy], pts[pts[:, 1].argmax()], pts[pts[:, 1].argmin()], pts[pts[:, 0].argmin()], pts[pts[:, 0].argmax()]])
    return np.concatenate([k, np.ones((5, 1))], 1)

def llantas_zeroshot(img, conf=0.15):
    r = yoloe().predict(img, conf=conf, imgsz=1024, device=DEVICE, verbose=False)[0]
    out = []
    if r.masks is None: return out
    H, W = r.orig_shape
    for b, s, poly in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.masks.xy):
        if len(poly) < 3: continue
        m = np.zeros((H, W), np.uint8); cv2.fillPoly(m, [poly.astype(np.int32)], 1)
        k = kps_de_mascara(m)
        if k is not None: k[:, 2] = s; out.append({'conf': float(s), 'box': b.tolist(), 'kps': k})
    return nms_simple(out)

altura_visible = lambda l: max(1.0, l['kps'][1, 1] - l['kps'][2, 1])

def limpiar_llantas(ls):
    res = []
    for l in sorted(ls, key=lambda l: -(l['box'][2]-l['box'][0]) * (l['box'][3]-l['box'][1])):
        if all(contenida(l['box'], r['box']) < UMBRAL_CONTENCION for r in res): res.append(l)
    return res

def analizar_ejes(caja, ls):
    # (ejes, vista). ejes=None en vista frontal/posterior o sin llantas — igual que el servidor
    ls = limpiar_llantas(ls)
    if not ls: return None, 'desconocida'
    hmax = max(altura_visible(l) for l in ls)
    cerc = [l for l in ls if altura_visible(l) >= UMBRAL_CERCANA * hmax]
    prop = (caja[2] - caja[0]) / max(1, caja[3] - caja[1])
    if len(cerc) == 2 and prop < 1.6:
        a, b = cerc
        if all((l['kps'][4, 0] - l['kps'][3, 0]) / altura_visible(l) < 0.65 for l in cerc) and \
           abs(a['kps'][1, 1] - b['kps'][1, 1]) < 0.15 * hmax:
            return None, 'frontal/posterior'
    xs = sorted(l['kps'][0, 0] for l in cerc)
    d = float(np.median([altura_visible(l) for l in cerc]))
    ejes = 1 + sum(1 for p, q in zip(xs, xs[1:]) if q - p > SEPARACION_EJE * d)
    return max(2, ejes), 'lateral'

## 3. Auto-etiquetado asistido (para corregir en Roboflow)
Suba un `.zip` con fotos/frames de vehículos (idealmente **vistas laterales**, ≥ 200). Se generan etiquetas
YOLO-pose que se importan a Roboflow para **corregirlas a mano** (un modelo entrenado solo con etiquetas
automáticas copia los errores del zero-shot).

In [ ]:
AUTOETIQUETAR = True
DIR_IMGS, DIR_AUTO = '/content/imagenes', '/content/autoetiquetado'
if AUTOETIQUETAR:
    from google.colab import files
    os.makedirs(DIR_IMGS, exist_ok=True)
    for n in files.upload():
        if n.lower().endswith('.zip'): os.system(f'unzip -qo "{n}" -d {DIR_IMGS}')
        else: shutil.move(n, DIR_IMGS)
    imgs = sorted(p for p in glob.glob(DIR_IMGS + '/**/*', recursive=True) if p.lower().endswith(('.jpg', '.jpeg', '.png')))
    shutil.rmtree(DIR_AUTO, ignore_errors=True); os.makedirs(DIR_AUTO + '/images'); os.makedirs(DIR_AUTO + '/labels')
    total = 0
    for i, ruta in enumerate(imgs):
        im = cv2.imread(ruta)
        if im is None: continue
        H, W = im.shape[:2]; base = f'{i:05d}'
        cv2.imwrite(f'{DIR_AUTO}/images/{base}.jpg', im)
        lineas = []
        for l in limpiar_llantas(llantas_zeroshot(im)):
            k = l['kps']; x1, y1, x2, y2 = l['box']
            x1, y1, x2, y2 = min(x1, k[:, 0].min()), min(y1, k[:, 1].min()), max(x2, k[:, 0].max()), max(y2, k[:, 1].max())
            v = [(x1 + x2) / 2 / W, (y1 + y2) / 2 / H, (x2 - x1) / W, (y2 - y1) / H]
            for x, y, _ in k: v += [min(max(x / W, 0), 1), min(max(y / H, 0), 1), 2]
            lineas.append('0 ' + ' '.join(f'{t:.6f}' if isinstance(t, float) else str(t) for t in v))
        open(f'{DIR_AUTO}/labels/{base}.txt', 'w').write('\n'.join(lineas)); total += len(lineas)
    open(f'{DIR_AUTO}/data.yaml', 'w').write(f'train: images\nval: images\nkpt_shape: [5, 3]\nflip_idx: {FLIP_IDX}\nnames:\n  0: llanta\n')
    os.system(f'cd /content && zip -qr autoetiquetado_llantas.zip autoetiquetado')
    print(len(imgs), 'imágenes ·', total, 'llantas auto-etiquetadas')
    files.download('/content/autoetiquetado_llantas.zip')

### En Roboflow
1. *Create Project → **Keypoint Detection***, clase `llanta`, esqueleto con 5 puntos en este orden:
   `centro, contacto, superior, izquierda, derecha` (conexiones centro→cada uno).
2. *Upload* → arrastre la carpeta `autoetiquetado` (formato YOLOv8 pose). Corrija cada llanta.
3. *Generate* (augmentations: flip horizontal **activa el intercambio izquierda/derecha**) → *Export YOLOv8*.

## 4. Descargar el dataset corregido desde Roboflow

In [ ]:
from google.colab import userdata
from roboflow import Roboflow
RF_WORKSPACE, RF_PROYECTO, RF_VERSION = 'tu-workspace', 'llantas-keypoints', 1   # <- cambie
ds = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY')).workspace(RF_WORKSPACE).project(RF_PROYECTO) \
        .version(RF_VERSION).download('yolov8')
DATA_YAML = f'{ds.location}/data.yaml'
cfg = yaml.safe_load(open(DATA_YAML))
cfg['kpt_shape'] = [5, 3]; cfg['flip_idx'] = FLIP_IDX
yaml.safe_dump(cfg, open(DATA_YAML, 'w')); print(cfg)

## 5. Entrenamiento YOLO11s-pose

In [ ]:
modelo = YOLO('yolo11s-pose.pt')        # el cabezal de 17 kps (humano) se adapta a 5
modelo.train(data=DATA_YAML, epochs=100, imgsz=640, batch=16, patience=25, fliplr=0.5,
             project='/content/runs/pose', name='m3', exist_ok=True, device=0)
best = YOLO('/content/runs/pose/m3/weights/best.pt')
val = best.val(data=DATA_YAML, split='valid', project='/content/runs/pose', name='m3_val', exist_ok=True)
print(f'mAP50 pose: {val.pose.map50:.4f} | mAP50-95 pose: {val.pose.map:.4f} | mAP50 box: {val.box.map50:.4f}')

## 6. MAE del conteo de ejes
* **Ejes reales**: se calculan con la misma regla sobre los keypoints **etiquetados a mano** (o desde
  `ejes_gt.csv` con columnas `imagen,ejes` si lo sube).
* **Ejes predichos**: misma regla sobre los keypoints predichos por el modelo.
* MAE = promedio |predichos − reales| en imágenes con vista lateral.

In [ ]:
def llantas_de_label(ruta_lbl, W, H):
    out = []
    for linea in open(ruta_lbl).read().split('\n'):
        v = linea.split()
        if len(v) < 5 + 15: continue
        cx, cy, bw, bh = map(float, v[1:5])
        k = np.array([[float(v[5+3*j]) * W, float(v[6+3*j]) * H, float(v[7+3*j])] for j in range(5)])
        out.append({'conf': 1.0, 'box': [(cx-bw/2)*W, (cy-bh/2)*H, (cx+bw/2)*W, (cy+bh/2)*H], 'kps': k})
    return out

def llantas_pred(img):
    r = best.predict(img, conf=0.25, device=DEVICE, verbose=False)[0]
    out = []
    if r.keypoints is not None and len(r.boxes):
        xy, kc = r.keypoints.xy.cpu().numpy(), r.keypoints.conf.cpu().numpy()
        for b, s, p, c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), xy, kc):
            out.append({'conf': float(s), 'box': b.tolist(), 'kps': np.concatenate([p, c[:, None]], 1)})
    return out

gt_csv = {}
if os.path.exists('/content/ejes_gt.csv'):
    gt_csv = dict(pd.read_csv('/content/ejes_gt.csv').values)

filas = []
for ruta in sorted(glob.glob(f'{ds.location}/valid/images/*')):
    img = cv2.imread(ruta); H, W = img.shape[:2]
    lbl = ruta.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    caja = [0, 0, W, H]
    real, vista = analizar_ejes(caja, llantas_de_label(lbl, W, H)) if os.path.exists(lbl) else (None, '-')
    real = gt_csv.get(os.path.basename(ruta), real)
    pred, _ = analizar_ejes(caja, llantas_pred(img))
    filas.append({'imagen': os.path.basename(ruta), 'vista': vista, 'ejes_reales': real, 'ejes_predichos': pred})
ej = pd.DataFrame(filas)
ok = ej.dropna(subset=['ejes_reales'])
ok = ok.assign(ejes_predichos=ok.ejes_predichos.fillna(2))      # el backend asume 2 si no es determinable
mae = (ok.ejes_reales - ok.ejes_predichos).abs().mean()
exact = (ok.ejes_reales == ok.ejes_predichos).mean() * 100
json.dump({'mAP50_pose': float(val.pose.map50), 'mAP50_95_pose': float(val.pose.map), 'mAP50_box': float(val.box.map50),
           'MAE_ejes': float(mae), 'exactitud_ejes_pct': float(exact), 'imagenes_evaluadas': int(len(ok))},
          open('/content/m3_metricas.json', 'w'), indent=2)
print(f'MAE ejes = {mae:.3f} · exactitud = {exact:.1f}% · n = {len(ok)}')
ej

## 7. Visualización (estilo imagen referencial M3)

In [ ]:
for ruta in sorted(glob.glob(f'{ds.location}/valid/images/*'))[:4]:
    img = cv2.imread(ruta); ls = limpiar_llantas(llantas_pred(img))
    ejes, vista = analizar_ejes([0, 0, img.shape[1], img.shape[0]], ls)
    xs = sorted(ls, key=lambda l: l['kps'][0, 0])
    for i, l in enumerate(xs, 1):
        c = tuple(map(int, l['kps'][0, :2])); cv2.circle(img, c, 12, (255, 255, 255), -1); cv2.circle(img, c, 9, (94, 63, 244), -1)
        cv2.putText(img, f'K{i}', (c[0] - 15, c[1] - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (94, 63, 244), 2)
    for a, b in zip(xs, xs[1:]):
        cv2.line(img, tuple(map(int, a['kps'][0, :2])), tuple(map(int, b['kps'][0, :2])), (94, 63, 244), 2)
    plt.figure(figsize=(10, 6)); plt.imshow(img[:, :, ::-1]); plt.title(f'{ejes} ejes · vista {vista}'); plt.axis('off'); plt.show()

## 8. Exportar `m3_llantas_pose.pt`

In [ ]:
shutil.copy('/content/runs/pose/m3/weights/best.pt', '/content/m3_llantas_pose.pt')
try:
    from google.colab import drive; drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/peaje-ia/pesos', exist_ok=True)
    shutil.copy('/content/m3_llantas_pose.pt', '/content/drive/MyDrive/peaje-ia/pesos/')
    shutil.copy('/content/m3_metricas.json', '/content/drive/MyDrive/peaje-ia/')
except Exception as e:
    print('Drive no montado:', e)
from google.colab import files
files.download('/content/m3_llantas_pose.pt'); files.download('/content/m3_metricas.json')